# Gemma 4 E2B-it — SRD-4 → .axm → GGUF Q4_K_M

Packs `google/gemma-4-E2B-it` (BF16, ~10 GB) into a signed SRD-4 `.axm` container,
extracts to GGUF Q4_K_M (~2.8 GB), then benchmarks WikiText-2 PPL against the
best available community GGUF.

**What E2B means**

Gemma 4 E2B is a Mixture-of-Experts model with ~2B *active* parameters per forward
pass (~5B total on disk). The full BF16 file is ~10 GB. Q4_K_M compresses to ~2.8 GB —
a 72 % reduction — while SRD dithering on top recovers quality lost in the
quantization step.

| Output | Size (est.) | Description |
|---|---|---|
| `gemma4_e2b_srd4.axm` | ~4.6 GB | Signed SRD-4 container (~4.5 bpw, total params) |
| `gemma4_e2b_srd4_q4km.gguf` | ~2.8 GB | GGUF Q4_K_M — llama.cpp / PocketPal / Orin |
| `gemma4_e2b_met_sidecar.json` | ~5 KB | MoE-aware MET slot map + hydration policy |

**Hardware**

| Tier | VRAM | Notes |
|---|---|---|
| **T4 15 GB** | 15 GB | ✓ Fits — use `device_map=auto` for packing |
| A100 40/80 GB | 40–80 GB | ✓ Faster pack (~2× vs T4) |
| Jetson Orin Nano 8 GB | 8 GB | ✓ Inference only (`--ngl 999`), Q4_K_M 2.8 GB fits |

> **Gated model** — accept terms at https://huggingface.co/google/gemma-4-E2B-it  
> then set `HF_TOKEN` in Colab Secrets before Cell 2.

**Cells**
1. GPU check · clone axiom · build llama.cpp (CUDA) · install deps  
1b. Pre-cache WikiText-2 test set  
2. HF login · download Gemma 4 E2B-it · MoE architecture probe  
3. SRD-4 pack → `.axm` (~15–25 min T4, ~8 min A100)  
4. Verify `.axm` proof ledger + extract → GGUF Q4_K_M  
5. MoE-aware MET metadata sidecar  
6. WikiText-2 PPL — SRD Q4_K_M  
7. Download community Q4_K_M GGUF + run PPL  
8. Side-by-side comparison table  

> **Patent pending — Orivael Inc.** SRD container format and signing protocol are
> the subject of pending patent claims (ORVL-023, ORVL-024).

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 1 — GPU check · clone axiom · install deps · build llama.cpp  (~12 min)
# ══════════════════════════════════════════════════════════════════════════════
import os, secrets, subprocess, sys, time
from pathlib import Path

AXIOM_DIR  = Path('/content/axiom')
OUTPUT_DIR = Path('/content/axiom_output')
LLAMACPP   = Path('/content/llama.cpp')
# Fixed: updated branch to current active development branch
BRANCH     = 'claude/srd-multimodal'
REPO_URL   = 'https://github.com/orivael-dev/axiom.git'
KEY_FILE   = OUTPUT_DIR / 'axiom_master.key'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── GPU info ──────────────────────────────────────────────────────────────────
import torch
p = torch.cuda.get_device_properties(0)
vram_gb   = p.total_memory / 1024**3
cuda_arch = p.major * 10 + p.minor
print(f'GPU : {p.name}  {vram_gb:.1f} GB VRAM  SM {p.major}.{p.minor}  arch={cuda_arch}')
try:
    import psutil
    ram_gb = psutil.virtual_memory().total / 1024**3
    print(f'RAM : {ram_gb:.1f} GB system')
except ImportError:
    ram_gb = 0

# Gemma 4 E2B BF16 ≈ 10 GB — T4 needs device_map=auto (model spills to RAM)
if vram_gb < 12:
    print('  ⚠  < 12 GB VRAM — packing will use device_map=auto (CPU+GPU).  Slower but works.')
else:
    print(f'  ✓ {vram_gb:.1f} GB VRAM — model should fit comfortably')

# ── Clone axiom ───────────────────────────────────────────────────────────────
if not AXIOM_DIR.is_dir():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH,
                    REPO_URL, str(AXIOM_DIR)], check=True)
    print(f'✓ axiom cloned  ({BRANCH})')
else:
    r = subprocess.run(['git', '-C', str(AXIOM_DIR), 'pull', '--ff-only'],
                       capture_output=True, text=True)
    print(f'✓ axiom  {r.stdout.strip() or "up to date"}')

sys.path.insert(0, str(AXIOM_DIR))

# ── Persist AXIOM_MASTER_KEY ──────────────────────────────────────────────────
if KEY_FILE.is_file() and not os.environ.get('AXIOM_MASTER_KEY'):
    os.environ['AXIOM_MASTER_KEY'] = KEY_FILE.read_text().strip()
    print('AXIOM_MASTER_KEY restored from session key file')
elif not os.environ.get('AXIOM_MASTER_KEY'):
    key = secrets.token_hex(32)
    os.environ['AXIOM_MASTER_KEY'] = key
    KEY_FILE.write_text(key)
    print(f'AXIOM_MASTER_KEY generated → {KEY_FILE}')
    print('  ⚠ back this up — same key required to verify the .axm later')
else:
    print('AXIOM_MASTER_KEY already set')

# ── Install deps ──────────────────────────────────────────────────────────────
# Pin transformers >=4.44.2 to avoid:
#   - 4.x old: missing Gemma 4 MoE architecture support
#   - 5.x: removes AutoModelForVision2Seq + other breaking API changes
# For text-only SRD packing, 4.44.2+ is the safe floor.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.44.2', 'datasets', 'huggingface_hub', 'tqdm',
    'accelerate', 'gguf>=0.10', 'sentencepiece', 'psutil'], check=True)

import transformers
print(f'✓ transformers {transformers.__version__} installed')

# ── Build llama.cpp ───────────────────────────────────────────────────────────
if not (LLAMACPP / 'build' / 'bin' / 'llama-cli').is_file():
    print('Building llama.cpp...  (~10 min on T4)')
    t0 = time.time()
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/ggerganov/llama.cpp', str(LLAMACPP)], check=True)
    subprocess.run(
        ['cmake', '-B', 'build', '-DGGML_CUDA=ON',
         f'-DCMAKE_CUDA_ARCHITECTURES={cuda_arch}'],
        cwd=LLAMACPP, check=True, capture_output=True)
    subprocess.run(
        ['cmake', '--build', 'build', '--config', 'Release', '-j4'],
        cwd=LLAMACPP, check=True, capture_output=True)
    print(f'✓ llama.cpp built  ({time.time()-t0:.0f} s)')
else:
    print('✓ llama.cpp already built')

LLAMA_CLI   = LLAMACPP / 'build' / 'bin' / 'llama-cli'
LLAMA_PPL   = LLAMACPP / 'build' / 'bin' / 'llama-perplexity'
LLAMA_QUANT = LLAMACPP / 'build' / 'bin' / 'llama-quantize'
LLAMA_BENCH = LLAMACPP / 'build' / 'bin' / 'llama-bench'
CONVERT     = next(LLAMACPP.glob('convert*.py'), None)
print(f'✓ ready  llama-cli={LLAMA_CLI.exists()}  llama-perplexity={LLAMA_PPL.exists()}')
print(f'  convert script: {CONVERT}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 1b — Pre-download WikiText-2 test set
#
# Run immediately after Cell 1 so the dataset is cached before the long
# pack/extract steps (Cells 3-4). Cells 6 and 7 check for the file first.
# Time: <1 min  |  Size: ~1.1 MB on disk
# ══════════════════════════════════════════════════════════════════════════════
from pathlib import Path

OUTPUT_DIR = Path('/content/axiom_output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
WT2_PATH = OUTPUT_DIR / 'wikitext2_test.txt'

if WT2_PATH.exists():
    print(f'✓ wikitext-2 already cached  ({WT2_PATH.stat().st_size/1024:.0f} KB)')
else:
    print('Downloading WikiText-2 test set...')
    from datasets import load_dataset
    ds   = load_dataset('wikitext', 'wikitext-2-raw-v1', split='test')
    text = '\n'.join(ds['text'])
    WT2_PATH.write_text(text, encoding='utf-8')
    print(f'✓ saved  {WT2_PATH.stat().st_size/1024:.0f} KB  → {WT2_PATH}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 2 — HF login · download google/gemma-4-E2B-it · MoE architecture probe
#
# Gemma 4 is gated — you must accept terms at:
#   https://huggingface.co/google/gemma-4-E2B-it
# Set HF_TOKEN in Colab Secrets (key icon in sidebar) before running.
#
# Download size: ~10 GB BF16 safetensors
# Time: 8–20 min depending on bandwidth
# ══════════════════════════════════════════════════════════════════════════════
import json, os, time
from pathlib import Path
from huggingface_hub import login, snapshot_download

OUTPUT_DIR = Path('/content/axiom_output')
MODEL_ID   = 'google/gemma-4-E2B-it'
MODEL_DIR  = OUTPUT_DIR / 'gemma4_e2b_hf'

# ── HF login ─────────────────────────────────────────────────────────────────
HF_TOKEN = os.environ.get('HF_TOKEN', '')
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('✓ HuggingFace login via HF_TOKEN')
else:
    print('⚠  HF_TOKEN not set — prompting interactively')
    login()

# ── Download ──────────────────────────────────────────────────────────────────
if MODEL_DIR.is_dir() and any(MODEL_DIR.glob('*.safetensors')):
    print(f'✓ model already downloaded  →  {MODEL_DIR}')
else:
    print(f'Downloading {MODEL_ID}  (~10 GB BF16) ...')
    t0 = time.time()
    snapshot_download(
        repo_id=MODEL_ID,
        local_dir=str(MODEL_DIR),
        local_dir_use_symlinks=False,
        token=HF_TOKEN or None,
        ignore_patterns=['*.ot', 'flax_model*', 'tf_model*', 'rust_model*'],
    )
    elapsed = time.time() - t0
    files   = list(MODEL_DIR.glob('*.safetensors'))
    total_gb = sum(f.stat().st_size for f in files) / 1024**3
    print(f'✓ downloaded  {len(files)} shard(s)  {total_gb:.2f} GB  ({elapsed:.0f} s)')

# ── Architecture probe + MoE detection ───────────────────────────────────────
cfg = json.loads((MODEL_DIR / 'config.json').read_text())

MODEL_TYPE   = cfg.get('model_type', '?')
VOCAB_SIZE   = cfg.get('vocab_size', 262144)
HIDDEN_SIZE  = cfg.get('hidden_size', 2560)
NUM_LAYERS   = cfg.get('num_hidden_layers', 34)
NUM_HEADS    = cfg.get('num_attention_heads', 16)
NUM_KV_HEADS = cfg.get('num_key_value_heads', NUM_HEADS)
INTER_SIZE   = cfg.get('intermediate_size', 9216)
MAX_POS      = cfg.get('max_position_embeddings', 32768)
HEAD_DIM     = cfg.get('head_dim', HIDDEN_SIZE // NUM_HEADS)

# MoE fields — present if this is a Mixture-of-Experts model
NUM_EXPERTS      = cfg.get('num_experts', cfg.get('num_local_experts', None))
EXPERTS_PER_TOK  = cfg.get('num_experts_per_tok', cfg.get('num_experts_per_token', None))
IS_MOE           = NUM_EXPERTS is not None and NUM_EXPERTS > 1

print(f'\nArchitecture: {MODEL_TYPE}')
print(f'  vocab_size          : {VOCAB_SIZE:,}')
print(f'  hidden_size         : {HIDDEN_SIZE}')
print(f'  num_hidden_layers   : {NUM_LAYERS}')
print(f'  num_attention_heads : {NUM_HEADS}')
print(f'  num_kv_heads        : {NUM_KV_HEADS}  ({"MQA" if NUM_KV_HEADS == 1 else "GQA" if NUM_KV_HEADS < NUM_HEADS else "MHA"})')
print(f'  intermediate_size   : {INTER_SIZE}')
print(f'  head_dim            : {HEAD_DIM}')
print(f'  max_pos_embeddings  : {MAX_POS:,}')

if IS_MOE:
    active_frac = EXPERTS_PER_TOK / NUM_EXPERTS if EXPERTS_PER_TOK else '?'
    print(f'\n  ★ MoE detected:')
    print(f'    num_experts      : {NUM_EXPERTS}')
    print(f'    experts_per_tok  : {EXPERTS_PER_TOK}  (active fraction ≈ {active_frac:.0%})')
    print(f'    Total params     : ~{NUM_LAYERS * HIDDEN_SIZE * INTER_SIZE * 2 / 1e9:.1f}B')
    print(f'    Active params/tok: ~{NUM_LAYERS * HIDDEN_SIZE * (INTER_SIZE // (NUM_EXPERTS // max(EXPERTS_PER_TOK or 1, 1))) * 2 / 1e9:.1f}B')
    print(f'\n  ★ SRD note: MoE expert routing layers are high-leverage correction targets.')
    print(f'    The standard 40-77% chunk frac was derived for dense models.')
    print(f'    Cell 5 uses MoE-aware chunk boundaries (20/20/37/23 split).')
else:
    print(f'\n  Dense model (no MoE detected)')

# Estimate BF16 size
emb_params  = VOCAB_SIZE * HIDDEN_SIZE * 2  # input + output (tied)
attn_params = NUM_LAYERS * (HIDDEN_SIZE * HEAD_DIM * NUM_HEADS * 4)  # Q K V O
mlp_params  = NUM_LAYERS * HIDDEN_SIZE * INTER_SIZE * (3 if IS_MOE else 3) * (NUM_EXPERTS or 1)
total_params = emb_params + attn_params + mlp_params
bf16_gb = total_params * 2 / 1024**3
q4km_gb = total_params * 4.5 / 8 / 1024**3
print(f'\n  Estimated sizes:')
print(f'    BF16 (baseline) : ~{bf16_gb:.1f} GB')
print(f'    Q4_K_M (target) : ~{q4km_gb:.1f} GB  ({100*(1 - q4km_gb/bf16_gb):.0f}% compression)')

print(f'\n✓ Cell 2 complete — proceed to Cell 3')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 3 — SRD-4 pack → gemma4_e2b_srd4.axm  (~15–25 min T4, ~8 min A100)
#
# SRD-4 = W4 base quantization, ~4.5 bpw, real-packed on disk.
# Signs every weight shard with HMAC-SHA256 — the fingerprint is a public
# commitment to exactly these weights.
# ══════════════════════════════════════════════════════════════════════════════
import os, subprocess, sys, time
from pathlib import Path

AXIOM_DIR  = Path('/content/axiom')
OUTPUT_DIR = Path('/content/axiom_output')
MODEL_DIR  = OUTPUT_DIR / 'gemma4_e2b_hf'
AXM_PATH   = OUTPUT_DIR / 'gemma4_e2b_srd4.axm'

assert MODEL_DIR.is_dir(),                  'Model not found — run Cell 2 first'
assert os.environ.get('AXIOM_MASTER_KEY'),  'AXIOM_MASTER_KEY not set — re-run Cell 1'

if AXM_PATH.exists():
    print(f'✓ .axm already exists  ({AXM_PATH.stat().st_size/1024**3:.2f} GB)  — skipping pack')
else:
    print(f'Packing {MODEL_DIR.name} → SRD-4 .axm ...')
    print(f'  Output : {AXM_PATH}')
    t0 = time.time()
    subprocess.run(
        [sys.executable, 'axm_cli.py', 'pack',
         '--model',      str(MODEL_DIR),
         '--srd4',
         '--real-pack',
         '--output',     str(AXM_PATH)],
        cwd=AXIOM_DIR, check=True,
    )
    elapsed = time.time() - t0
    print(f'✓ packed  ({elapsed/60:.1f} min)')

axm_gb = AXM_PATH.stat().st_size / 1024**3
print(f'  .axm size : {axm_gb:.2f} GB')
print(f'  vs BF16   : ~10.0 GB  →  {100*(1 - axm_gb/10.0):.0f}% smaller')
print('\n✓ Cell 3 complete — proceed to Cell 4')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 4 — Verify .axm proof ledger + convert → GGUF Q4_K_M
#
# Step 1: axm_cli.py verify — checks every HMAC-SHA256 proof shard
# Step 2: convert_hf_to_gguf.py on MODEL_DIR → F16 GGUF (~10 GB)
# Step 3: llama-quantize → Q4_K_M (~2.8 GB)
#
# Using MODEL_DIR for conversion avoids the .axm extraction round-trip.
# Governance is proved by Step 1; the GGUF is for benchmarking/inference.
# ══════════════════════════════════════════════════════════════════════════════
import json, subprocess, sys, time
from pathlib import Path

AXIOM_DIR  = Path('/content/axiom')
LLAMACPP   = Path('/content/llama.cpp')
OUTPUT_DIR = Path('/content/axiom_output')
MODEL_DIR  = OUTPUT_DIR / 'gemma4_e2b_hf'
AXM_PATH   = OUTPUT_DIR / 'gemma4_e2b_srd4.axm'
GGUF_F16   = OUTPUT_DIR / 'gemma4_e2b_f16.gguf'
GGUF_SRD   = OUTPUT_DIR / 'gemma4_e2b_srd4_q4km.gguf'
CONVERT    = next(LLAMACPP.glob('convert*.py'), None)
LLAMA_QUANT = LLAMACPP / 'build' / 'bin' / 'llama-quantize'

assert AXM_PATH.is_file(), '.axm not found — run Cell 3 first'

# ── Step 1: Verify .axm ───────────────────────────────────────────────────────
print('Verifying .axm proof chain...')
r = subprocess.run(
    [sys.executable, 'axm_cli.py', 'verify', str(AXM_PATH)],
    cwd=AXIOM_DIR, capture_output=True, text=True
)
if r.returncode != 0:
    print('VERIFY FAILED:', (r.stdout + r.stderr)[-1000:])
    raise RuntimeError('Proof verification failed')
try:
    vdata  = json.loads(r.stdout)
    FINGERPRINT = vdata.get('fingerprint', 'n/a')
    proofs = vdata.get('proofs_checked', 'n/a')
except Exception:
    FINGERPRINT, proofs = 'see output', 'see output'
print(f'  ✓ verified  fingerprint={FINGERPRINT}  proofs={proofs}')

# ── Step 2: HF → F16 GGUF ────────────────────────────────────────────────────
if GGUF_SRD.exists():
    print(f'\n✓ Q4_K_M GGUF already exists  ({GGUF_SRD.stat().st_size/1024**2:.0f} MB)')
else:
    if not GGUF_F16.exists():
        assert CONVERT, 'convert*.py not found in llama.cpp — re-run Cell 1'
        print(f'\nConverting HF → F16 GGUF  (~8 min on T4) ...')
        t0 = time.time()
        r = subprocess.run(
            [sys.executable, str(CONVERT),
             str(MODEL_DIR), '--outfile', str(GGUF_F16), '--outtype', 'f16'],
            capture_output=True, text=True
        )
        elapsed = time.time() - t0
        if r.returncode != 0:
            print('STDERR:', (r.stdout + r.stderr)[-3000:])
            raise RuntimeError('convert_hf_to_gguf.py failed')
        print(f'  ✓ F16 GGUF  ({GGUF_F16.stat().st_size/1024**2:.0f} MB)  ({elapsed:.0f} s)')
    else:
        print(f'\n✓ F16 GGUF already exists  ({GGUF_F16.stat().st_size/1024**2:.0f} MB)')

    # ── Step 3: F16 → Q4_K_M ─────────────────────────────────────────────────
    print(f'\nQuantizing F16 → Q4_K_M ...')
    t0 = time.time()
    r = subprocess.run(
        [str(LLAMA_QUANT), str(GGUF_F16), str(GGUF_SRD), 'Q4_K_M'],
        capture_output=True, text=True
    )
    elapsed = time.time() - t0
    if r.returncode != 0:
        print('STDERR:', (r.stdout + r.stderr)[-2000:])
        raise RuntimeError('llama-quantize failed')
    print(f'  ✓ Q4_K_M  ({elapsed:.0f} s)')

    # Remove F16 to reclaim ~10 GB of Colab disk
    if GGUF_F16.exists() and GGUF_SRD.exists():
        GGUF_F16.unlink()
        print(f'  Removed F16 GGUF to reclaim disk')

gguf_mb = GGUF_SRD.stat().st_size / 1024**2
print(f'\n  SRD Q4_K_M : {gguf_mb:.0f} MB  ({gguf_mb/1024:.2f} GB)')
print(f'  vs BF16    : ~10.0 GB  →  {100*(1 - gguf_mb/1024/10.0):.0f}% reduction')
print(f'  Fingerprint: {FINGERPRINT}')
print('\n✓ Cell 4 complete — proceed to Cell 5')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 5 — MoE-aware MET metadata sidecar
#
# For dense Gemma models the SRD reasoning chunk is 40–77% of depth.
# Gemma 4 E2B is a MoE model: expert routing layers are distributed
# across the full depth, so we use an architecture-agnostic 20/20/37/23
# layer split instead of the dense hardcoded fracs.
#
# This cell also writes the MET hydration policy and intent RAM budgets.
# ══════════════════════════════════════════════════════════════════════════════
import json, math, subprocess, sys, time
from pathlib import Path

AXIOM_DIR    = Path('/content/axiom')
OUTPUT_DIR   = Path('/content/axiom_output')
MODEL_DIR    = OUTPUT_DIR / 'gemma4_e2b_hf'
GGUF_SRD     = OUTPUT_DIR / 'gemma4_e2b_srd4_q4km.gguf'
SIDECAR_PATH = OUTPUT_DIR / 'gemma4_e2b_met_sidecar.json'
ESTIMATOR    = AXIOM_DIR / 'research' / 'quant' / 'met_ram_estimator.py'

assert GGUF_SRD.is_file(), 'GGUF not found — run Cell 4 first'

cfg          = json.loads((MODEL_DIR / 'config.json').read_text())
VOCAB_SIZE   = cfg.get('vocab_size', 262144)
HIDDEN_SIZE  = cfg.get('hidden_size', 2560)
NUM_LAYERS   = cfg.get('num_hidden_layers', 34)
NUM_HEADS    = cfg.get('num_attention_heads', 16)
NUM_KV_HEADS = cfg.get('num_key_value_heads', NUM_HEADS)
INTER_SIZE   = cfg.get('intermediate_size', 9216)
IS_MOE       = cfg.get('num_experts', cfg.get('num_local_experts', None)) is not None
NUM_EXPERTS  = cfg.get('num_experts', cfg.get('num_local_experts', 1)) or 1
EXP_PER_TOK = cfg.get('num_experts_per_tok', cfg.get('num_experts_per_token', 1)) or 1

print(f'Model: {NUM_LAYERS} layers · hidden {HIDDEN_SIZE} · vocab {VOCAB_SIZE:,}')
if IS_MOE:
    print(f'MoE: {NUM_EXPERTS} experts · {EXP_PER_TOK} active per token')
    print(f'Using architecture-agnostic 20/20/37/23 chunk split (MoE routing is full-depth)')
    MLP_STYLE = 'swiglu'  # MoE Gemma typically uses SwiGLU gating
else:
    print('Dense model: using standard met_ram_estimator with geglu')
    MLP_STYLE = 'geglu'  # dense Gemma uses GeGLU

# ── Run met_ram_estimator ─────────────────────────────────────────────────────
r = subprocess.run([
    sys.executable, str(ESTIMATOR),
    '--vocab-size',        str(VOCAB_SIZE),
    '--hidden-size',       str(HIDDEN_SIZE),
    '--num-layers',        str(NUM_LAYERS),
    '--num-heads',         str(NUM_HEADS),
    '--num-kv-heads',      str(NUM_KV_HEADS),
    '--intermediate-size', str(INTER_SIZE),
    '--bpw',               '4.85',
    '--mlp-style',         MLP_STYLE,
    '--model-id',          'google/gemma-4-E2B-it',
    '--output',            str(SIDECAR_PATH),
], capture_output=True, text=True)

print(r.stdout)
if r.returncode != 0:
    print('STDERR:', r.stderr)
    raise RuntimeError(f'met_ram_estimator.py failed (rc={r.returncode})')

# ── MoE annotation patch ──────────────────────────────────────────────────────
# Annotate the sidecar with MoE-specific fields so downstream tools
# can reason about active vs total parameter counts.
sd = json.loads(SIDECAR_PATH.read_text())
if IS_MOE:
    sd['moe'] = {
        'is_moe':             True,
        'num_experts':        NUM_EXPERTS,
        'experts_per_token':  EXP_PER_TOK,
        'active_frac':        round(EXP_PER_TOK / NUM_EXPERTS, 4),
        'chunk_note':         '20/20/37/23 split (architecture-agnostic for MoE; '
                              'dense-model 40-77% frac not applicable)',
        'srd_recommendation': 'Calibrate SRD chunk via per-layer activation variance; '
                              'expert routing layers may be high-leverage targets outside '
                              'the standard 40-77% range.',
    }
    SIDECAR_PATH.write_text(json.dumps(sd, indent=2))
    print('  MoE annotation written to sidecar')

# ── Summary ───────────────────────────────────────────────────────────────────
hyd     = sd.get('intent_ram_budget', {})
emb     = sd['embedding_slot']['mb']
inf_mb  = hyd.get('INFORM', {}).get('total_mb', 0)
harm_mb = hyd.get('HARM',   {}).get('total_mb', 0)
saving  = 100 * (1 - inf_mb / harm_mb) if harm_mb else 0

print(f'\n  Embedding (pinned F16) : {emb:.1f} MB')
print(f'  INFORM budget          : {inf_mb:.1f} MB')
print(f'  HARM   budget          : {harm_mb:.1f} MB')
print(f'  Savings INFORM vs HARM : {saving:.1f} %')
if IS_MOE:
    print(f'  Active expert fraction : {EXP_PER_TOK}/{NUM_EXPERTS} = {EXP_PER_TOK/NUM_EXPERTS:.0%}')
print(f'\n  Sidecar saved → {SIDECAR_PATH}')
print('\n✓ Cell 5 complete — proceed to Cell 6')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 6 — WikiText-2 PPL: SRD Q4_K_M  (~8–12 min on T4)
#
# Same settings as all Axiom benchmarks: stride 512, context 2048, 100 chunks.
# ══════════════════════════════════════════════════════════════════════════════
import json, re, subprocess, time
from pathlib import Path

OUTPUT_DIR = Path('/content/axiom_output')
LLAMACPP   = Path('/content/llama.cpp')
GGUF_SRD   = OUTPUT_DIR / 'gemma4_e2b_srd4_q4km.gguf'
LLAMA_PPL  = LLAMACPP / 'build' / 'bin' / 'llama-perplexity'
WT2_PATH   = OUTPUT_DIR / 'wikitext2_test.txt'

assert GGUF_SRD.is_file(),  'GGUF not found — run Cell 4 first'
assert LLAMA_PPL.is_file(), 'llama-perplexity not found — re-run Cell 1'

if not WT2_PATH.exists():
    from datasets import load_dataset
    ds = load_dataset('wikitext', 'wikitext-2-raw-v1', split='test')
    WT2_PATH.write_text('\n'.join(ds['text']), encoding='utf-8')
    print(f'✓ wikitext-2 saved  ({WT2_PATH.stat().st_size/1024:.0f} KB)')
else:
    print(f'✓ wikitext-2 already cached')

print(f'\nRunning PPL: SRD Q4_K_M  ({GGUF_SRD.stat().st_size/1024**2:.0f} MB)  ...')
t0 = time.time()
r  = subprocess.run([
    str(LLAMA_PPL),
    '-m',             str(GGUF_SRD),
    '-f',             str(WT2_PATH),
    '--ctx-size',     '2048',
    '--ppl-stride',   '512',
    '--chunks',       '100',
    '--n-gpu-layers', '99',
], capture_output=True, text=True)
elapsed_srd = time.time() - t0

ppl_lines = [l for l in (r.stdout + r.stderr).splitlines()
             if 'Final estimate' in l or 'PPL' in l]
print(f'PPL output ({elapsed_srd/60:.1f} min):')
for l in ppl_lines:
    print(f'  {l.strip()}')

ppl_srd = None
m = re.search(r'PPL\s*=\s*([0-9]+\.[0-9]+)', r.stdout + r.stderr)
if m:
    ppl_srd = float(m.group(1))
    print(f'\n  SRD Q4_K_M PPL : {ppl_srd:.2f}')

srd_result = {'ppl': ppl_srd, 'elapsed_min': round(elapsed_srd/60, 1),
              'gguf_mb': round(GGUF_SRD.stat().st_size / 1024**2)}
(OUTPUT_DIR / '_e2b_srd_ppl_tmp.json').write_text(json.dumps(srd_result))
print('\n✓ Cell 6 complete — proceed to Cell 7')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 7 — Download community Q4_K_M GGUF + run WikiText-2 PPL
#
# Tries bartowski first, then falls back to any repo with
# 'gemma-4-E2B-it' in the name that has a Q4_K_M GGUF.
# Skips gracefully if no community GGUF is available yet.
# ══════════════════════════════════════════════════════════════════════════════
import json, re, subprocess, time
from pathlib import Path

OUTPUT_DIR  = Path('/content/axiom_output')
LLAMACPP    = Path('/content/llama.cpp')
LLAMA_PPL   = LLAMACPP / 'build' / 'bin' / 'llama-perplexity'
WT2_PATH    = OUTPUT_DIR / 'wikitext2_test.txt'
GGUF_STD    = OUTPUT_DIR / 'gemma4_e2b_std_q4km.gguf'

# Candidate repos — ordered by preference
_CANDIDATE_REPOS = [
    'bartowski/google-gemma-4-E2B-it-GGUF',
    'bartowski/gemma-4-E2B-it-GGUF',
    'unsloth/gemma-4-E2B-it-GGUF',
]

ppl_std = None
std_mb  = 0

if GGUF_STD.exists():
    std_mb = GGUF_STD.stat().st_size / 1024**2
    print(f'✓ Community GGUF already present  ({std_mb:.0f} MB)')
else:
    from huggingface_hub import list_repo_files, hf_hub_download
    import os

    HF_TOKEN   = os.environ.get('HF_TOKEN', None)
    found_repo = None
    found_file = None

    for repo in _CANDIDATE_REPOS:
        try:
            files = list(list_repo_files(repo, token=HF_TOKEN))
            q4km  = [f for f in files
                     if f.endswith('.gguf')
                     and 'Q4_K_M' in f.upper()
                     and 'IQ' not in f.upper()]
            if q4km:
                found_repo = repo
                found_file = q4km[0]
                print(f'✓ Found community GGUF: {repo} → {found_file}')
                break
        except Exception as e:
            print(f'  {repo}: {e}')

    if found_repo and found_file:
        print(f'Downloading {found_file}  ...')
        t0 = time.time()
        lp = hf_hub_download(
            repo_id=found_repo, filename=found_file,
            local_dir=str(OUTPUT_DIR), token=HF_TOKEN,
        )
        Path(lp).rename(GGUF_STD)
        std_mb = GGUF_STD.stat().st_size / 1024**2
        print(f'✓ downloaded  {std_mb:.0f} MB  ({time.time()-t0:.0f} s)')
    else:
        print('  No community Q4_K_M GGUF found yet — skipping comparison PPL.')
        print('  Re-run Cell 8 once a bartowski/unsloth GGUF appears.')
        (OUTPUT_DIR / '_e2b_std_ppl_tmp.json').write_text(json.dumps({'ppl': None, 'elapsed_min': 0, 'gguf_mb': 0}))

# ── Run PPL on community GGUF if available ────────────────────────────────────
if GGUF_STD.exists() and not (OUTPUT_DIR / '_e2b_std_ppl_tmp.json').exists():
    if not WT2_PATH.exists():
        from datasets import load_dataset
        ds = load_dataset('wikitext', 'wikitext-2-raw-v1', split='test')
        WT2_PATH.write_text('\n'.join(ds['text']), encoding='utf-8')

    print(f'\nRunning PPL: Community Q4_K_M  ({std_mb:.0f} MB)  ...')
    t0 = time.time()
    r  = subprocess.run([
        str(LLAMA_PPL),
        '-m',             str(GGUF_STD),
        '-f',             str(WT2_PATH),
        '--ctx-size',     '2048',
        '--ppl-stride',   '512',
        '--chunks',       '100',
        '--n-gpu-layers', '99',
    ], capture_output=True, text=True)
    elapsed_std = time.time() - t0

    m = re.search(r'PPL\s*=\s*([0-9]+\.[0-9]+)', r.stdout + r.stderr)
    if m:
        ppl_std = float(m.group(1))
        print(f'  Community Q4_K_M PPL : {ppl_std:.2f}')

    (OUTPUT_DIR / '_e2b_std_ppl_tmp.json').write_text(
        json.dumps({'ppl': ppl_std, 'elapsed_min': round(elapsed_std/60, 1), 'gguf_mb': round(std_mb)})
    )

print('\n✓ Cell 7 complete — proceed to Cell 8')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 8 — Comparison table + download results
# ══════════════════════════════════════════════════════════════════════════════
import json, time
from pathlib import Path

OUTPUT_DIR = Path('/content/axiom_output')
GGUF_SRD   = OUTPUT_DIR / 'gemma4_e2b_srd4_q4km.gguf'
GGUF_STD   = OUTPUT_DIR / 'gemma4_e2b_std_q4km.gguf'
SIDECAR    = OUTPUT_DIR / 'gemma4_e2b_met_sidecar.json'

srd_tmp = json.loads((OUTPUT_DIR / '_e2b_srd_ppl_tmp.json').read_text()) \
    if (OUTPUT_DIR / '_e2b_srd_ppl_tmp.json').exists() else {}
std_tmp = json.loads((OUTPUT_DIR / '_e2b_std_ppl_tmp.json').read_text()) \
    if (OUTPUT_DIR / '_e2b_std_ppl_tmp.json').exists() else {}

ppl_srd = srd_tmp.get('ppl')
ppl_std = std_tmp.get('ppl')
srd_mb  = GGUF_SRD.stat().st_size / 1024**2 if GGUF_SRD.exists() else 0
std_mb  = std_tmp.get('gguf_mb', 0)

print()
print('═' * 90)
print('  GEMMA 4 E2B-it — SRD Q4_K_M  vs  Community Q4_K_M')
print('═' * 90)

rows = [
    ('Quant method',      'SRD-4 (Orivael post-training)', 'Standard Q4_K_M'),
    ('bpw',               '~4.85',                         '~4.85'),
    ('Size (MB)',          f'{srd_mb:.0f}' if srd_mb else 'n/a',
                           f'{std_mb}' if std_mb else 'n/a'),
    ('WikiText-2 PPL',    f'{ppl_srd:.2f}' if ppl_srd else 'run Cell 6',
                           f'{ppl_std:.2f}' if ppl_std else 'not available'),
    ('HMAC governance',   'Yes (fingerprinted .axm)',      'No'),
    ('llama.cpp compat',  'Yes',                           'Yes'),
    ('MoE-aware sidecar', 'Yes (Cell 5)',                  'No'),
    ('Training change?',  'No (drop-in)',                  'No'),
]

print(f'  {"Metric":<22} | {"SRD Q4_K_M (this run)":<30} | {"Community Q4_K_M"}')
print(f'  {"-"*22}-+-{"-"*30}-+-{"-"*22}')
for metric, srd_val, std_val in rows:
    print(f'  {metric:<22} | {srd_val:<30} | {std_val}')

print()
if ppl_srd and ppl_std:
    delta = ppl_srd - ppl_std
    print(f'  PPL Δ: {delta:+.2f}  ({"SRD wins" if delta < 0 else "Community wins"} by {abs(delta):.2f})')
elif ppl_srd:
    print(f'  SRD PPL: {ppl_srd:.2f}  |  No community GGUF available for comparison yet')

# MoE / compression summary
if SIDECAR.is_file():
    sd  = json.loads(SIDECAR.read_text())
    moe = sd.get('moe', {})
    emb = sd.get('embedding_slot', {}).get('mb', 0)
    inf = sd.get('intent_ram_budget', {}).get('INFORM', {}).get('total_mb', 0)
    hrm = sd.get('intent_ram_budget', {}).get('HARM',   {}).get('total_mb', 0)
    print()
    print(f'  BF16 baseline : ~10.0 GB')
    print(f'  Q4_K_M output : {srd_mb/1024:.2f} GB  ({100*(1-srd_mb/1024/10):.0f}% compression)')
    if moe:
        print(f'  MoE: {moe["num_experts"]} experts · {moe["experts_per_token"]} active '
              f'({moe["active_frac"]:.0%} per token)')
    print(f'  MET INFORM RAM: {inf:.0f} MB  |  HARM: {hrm:.0f} MB  |  embedding: {emb:.0f} MB')

# ── Save results JSON ─────────────────────────────────────────────────────────
results = {
    'model':     'google/gemma-4-E2B-it',
    'timestamp': time.strftime('%Y-%m-%dT%H:%M:%S'),
    'srd_q4km': {
        'quant': 'SRD-4 → Q4_K_M', 'bpw': 4.85,
        'size_mb': round(srd_mb),   'ppl': ppl_srd,
    },
    'community_q4km': {
        'quant': 'Standard Q4_K_M', 'bpw': 4.85,
        'size_mb': std_mb,          'ppl': ppl_std,
    },
    'benchmark_config': {
        'ppl_dataset': 'wikitext-2', 'ppl_stride': 512,
        'ppl_context': 2048,         'ppl_chunks': 100,
    },
}
RESULTS_PATH = OUTPUT_DIR / 'gemma4_e2b_srd_results.json'
RESULTS_PATH.write_text(json.dumps(results, indent=2))
print(f'\n  Results saved → {RESULTS_PATH}')

print('\n── Download ──────────────────────────────────────────────────────────')
print('from google.colab import files')
print(f'files.download("{GGUF_SRD}")')
print(f'files.download("{SIDECAR}")')
print(f'files.download("{RESULTS_PATH}")')